<a href="https://colab.research.google.com/github/Linux-Server/Transformers/blob/main/Google-Bert-Base-Cased/02_PEFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


check_point = "facebook/opt-350m"


pipe = pipeline("text-generation", model=check_point)





Device set to use cuda:0


In [ ]:
pipe("I love you", max_new_tokens= 60)

[{'generated_text': 'I love you.\nI love you too, but I have a confession. I have a girlfriend.'}]

In [ ]:
model = AutoModelForCausalLM.from_pretrained(check_point)

In [ ]:
from torchinfo import summary

summary(model)

Layer (type:depth-idx)                             Param #
OPTForCausalLM                                     --
├─OPTModel: 1-1                                    --
│    └─OPTDecoder: 2-1                             --
│    │    └─Embedding: 3-1                         25,739,264
│    │    └─OPTLearnedPositionalEmbedding: 3-2     2,099,200
│    │    └─Linear: 3-3                            524,288
│    │    └─Linear: 3-4                            524,288
│    │    └─ModuleList: 3-5                        302,309,376
├─Linear: 1-2                                      25,739,264
Total params: 356,935,680
Trainable params: 356,935,680
Non-trainable params: 0

In [ ]:
# create peft config
from peft import LoraConfig, TaskType

peft_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05)

In [ ]:
from peft import get_peft_model

lora_model = get_peft_model(model, peft_config)

In [ ]:
lora_model.print_trainable_parameters()

trainable params: 1,572,864 || all params: 332,769,280 || trainable%: 0.4727


In [ ]:
## Fine the model for intruction tuning with
from datasets import load_dataset


raw = load_dataset("vicgalle/alpaca-gpt4")

raw = raw["train"].select(range(10000))
raw

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 10000
})

In [ ]:
def format_alpaca(example):
    inst, inp, out = example["instruction"].strip(), example["input"].strip(), example["output"].strip()
    if inp:
        prompt = f"<|user|>\n{inst}\n\n### Input:\n{inp}\n<|assistant|>\n"
    else:
        prompt = f"<|user|>\n{inst}\n<|assistant|>\n"
    full = prompt + out + "</s>"
    return {"text": full}





In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-350m", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    prompt = format_alpaca(example)["text"]
    tokens = tokenizer(prompt, truncation=True, max_length=1024, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tok_ds = raw.map(tokenize, remove_columns=raw.column_names, num_proc=4)



Map (num_proc=4):   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

base = AutoModelForCausalLM.from_pretrained(
    "facebook/opt-350m",
    torch_dtype="float16",
    device_map="auto"
)

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],   # OPT projection layers
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()


trainable params: 1,572,864 || all params: 332,769,280 || trainable%: 0.4727


In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./opt350m-alpaca-chat",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,     # effective 16
    num_train_epochs=3,
    fp16=True,
    learning_rate=2e-4,                # LoRA can take a higher LR
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=1,
    optim="adamw_torch"
)

trainer = Trainer(model=model, args=args, train_dataset=tok_ds)


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
trainer.train()

Step,Training Loss
50,5.093300
100,0.383700
150,0.352300
200,0.340000
250,0.325700
300,0.317400
350,0.319200
400,0.316200
450,0.341200
500,0.310800


TrainOutput(global_step=1875, training_loss=0.4454682217915853, metrics={'train_runtime': 2551.5467, 'train_samples_per_second': 11.758, 'train_steps_per_second': 0.735, 'total_flos': 5.620484800512e+16, 'train_loss': 0.4454682217915853, 'epoch': 3.0})

In [ ]:
def chat(instruction, input_text=None, max_new_tokens=256):
    if input_text:
        prompt = f"<|user|>\n{instruction}\n\n### Input:\n{input_text}\n<|assistant|>\n"
    else:
        prompt = f"<|user|>\n{instruction}\n<|assistant|>\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("<|assistant|>\n")[-1].strip()



In [ ]:
chat("Who are you?")

'I am an AI voice assistant, but I am not a person.'

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
model.save_pretrained("./opt350m-alpaca-10k-instruction-tune-3")

In [ ]:
model.push_to_hub("opt350m-alpaca-10k-instruction-tune-3")

adapter_model.safetensors:   0%|          | 0.00/6.30M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/sachin6624/opt350m-alpaca-10k-instruction-tune-3/commit/2ae23e25d663abe7074838b4247dfc299f7ffe8b', commit_message='Upload model', commit_description='', oid='2ae23e25d663abe7074838b4247dfc299f7ffe8b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sachin6624/opt350m-alpaca-10k-instruction-tune-3', endpoint='https://huggingface.co', repo_type='model', repo_id='sachin6624/opt350m-alpaca-10k-instruction-tune-3'), pr_revision=None, pr_num=None)